In [ ]:
# FoodSAM summarization helpers (test utilities)
import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional

BACKGROUND_CATEGORY_IDS = {"0"}
BACKGROUND_CATEGORY_NAMES = {"background"}


@dataclass
class FoodItemSummary:
    label: str
    count: int
    mask_ratio_sum: float

    @property
    def mask_ratio_mean(self) -> float:
        if self.count == 0:
            return 0.0
        return self.mask_ratio_sum / self.count

    def to_serializable(self) -> Dict[str, object]:
        return {
            "label": self.label,
            "count": int(self.count),
            "mask_ratio_sum": round(float(self.mask_ratio_sum), 6),
            "mask_ratio_mean": round(float(self.mask_ratio_mean), 6),
        }


@dataclass
class DishSummary:
    dish_id: str
    items: List[FoodItemSummary]
    total_food_segments: int
    total_food_mask_ratio: float
    sam_label_file: str

    def to_serializable(self) -> Dict[str, object]:
        return {
            "dish_id": self.dish_id,
            "total_food_segments": int(self.total_food_segments),
            "total_food_mask_ratio": round(float(self.total_food_mask_ratio), 6),
            "items": [item.to_serializable() for item in self.items],
            "sam_label_file": self.sam_label_file,
        }


def _is_background(row: Dict[str, str]) -> bool:
    category_id = (row.get("category_id") or "").strip()
    category_name = (row.get("category_name") or "").strip().lower()
    if category_id in BACKGROUND_CATEGORY_IDS:
        return True
    if category_name in BACKGROUND_CATEGORY_NAMES:
        return True
    return False


def _parse_mask_ratio(value: Optional[str]) -> float:
    try:
        return float(value)
    except (TypeError, ValueError):
        return 0.0


def summarize_dish_instances(label_file: Path, skip_background: bool = True) -> Optional[DishSummary]:
    counts: Dict[str, int] = {}
    mask_totals: Dict[str, float] = {}

    with label_file.open("r", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            if skip_background and _is_background(row):
                continue

            label = (row.get("category_name") or "").strip()
            if not label:
                continue

            counts[label] = counts.get(label, 0) + 1
            mask_totals[label] = mask_totals.get(label, 0.0) + _parse_mask_ratio(
                row.get("mask_count_ratio")
            )

    if not counts:
        return None

    dish_id = label_file.parent.parent.name
    items = [
        FoodItemSummary(label=label, count=counts[label], mask_ratio_sum=mask_totals[label])
        for label in sorted(counts)
    ]

    total_segments = sum(item.count for item in items)
    total_mask_ratio = sum(item.mask_ratio_sum for item in items)

    return DishSummary(
        dish_id=dish_id,
        items=items,
        total_food_segments=total_segments,
        total_food_mask_ratio=total_mask_ratio,
        sam_label_file=str(label_file),
    )


def summaries_to_dataframe(summaries):
    import pandas as pd  # type: ignore

    records = []
    for summary in summaries:
        for item in summary.items:
            records.append(
                {
                    "dish_id": summary.dish_id,
                    "label": item.label,
                    "count": int(item.count),
                    "mask_ratio_sum": round(float(item.mask_ratio_sum), 6),
                    "mask_ratio_mean": round(float(item.mask_ratio_mean), 6),
                    "total_food_segments": int(summary.total_food_segments),
                    "total_food_mask_ratio": round(float(summary.total_food_mask_ratio), 6),
                }
            )

    return pd.DataFrame.from_records(records)


In [ ]:
from pathlib import Path

candidate_roots = [
    Path('/media/chahar/48e17169-ad03-49a2-8ad7-9f071aaf3dde/FoodSAM_nutrition5k_outputs'),
    Path('data/FoodSAM_nutrition5k_outputs'),
    Path('../data/FoodSAM_nutrition5k_outputs'),
    Path('/data/FoodSAM_nutrition5k_outputs'),
    Path('FoodSAM_nutrition5k_outputs'),
    Path('../FoodSAM_nutrition5k_outputs'),
    Path('FoodSAM/nutrition5k_foodSAM_outputs'),
    Path('../FoodSAM/nutrition5k_foodSAM_outputs'),
    Path('FoodSAM/nutrition50_foodSAM_outputs'),
    Path('../FoodSAM/nutrition50_foodSAM_outputs'),
]

label_files = []
foodsam_root = None
for candidate in candidate_roots:
    if candidate.exists():
        label_files = sorted(candidate.glob('*/sam_mask_label/semantic_masks_category.txt'))
        if label_files:
            foodsam_root = candidate.resolve()
            break

if not label_files or foodsam_root is None:
    raise FileNotFoundError('Could not locate any semantic_masks_category.txt files. Update the candidate paths.')

print(f'Using FoodSAM outputs from: {foodsam_root}')
print(f'Found {len(label_files)} sam_mask_label files.')
preview_limit = min(20, len(label_files))
print(f'Preparing summaries for the first {preview_limit} dishes.')


In [ ]:
subset_summaries = []
for label_path in label_files[:preview_limit]:
    summary = summarize_dish_instances(label_path)
    if summary is not None:
        subset_summaries.append(summary)

print(f'Computed summaries for {len(subset_summaries)} dishes.')
[s.to_serializable() for s in subset_summaries[:5]]

In [ ]:
try:
    summaries_to_dataframe(subset_summaries)
except ImportError:
    print('Install pandas to view a tabular summary of the dish-level counts.')